In [1]:
pip install tensorflow pandas numpy scikit-learn matplotlib jupyter

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
model_df = pd.read_csv("clean_final_data.csv")

model_df["date"] = pd.to_datetime(model_df["date"])

In [4]:
external_cols = [
    "crude_oil_avg_usd",
    "energy_index",
    "non_energy_index",
    "agriculture_index",
    "food_index",
    "fertilizers_index",
    "metals_minerals_index",
    "base_metals_index",
    "precious_metals_index",
    "commodity_total_index"
]

In [5]:
external_cols = [
    "crude_oil_avg_usd", "energy_index", "non_energy_index",
    "food_index", "fertilizers_index", "base_metals_index",
    "precious_metals_index", "fed_funds_rate", "usd_broad_index",
    "us_industrial_production", "us_cpi", "us_10y_treasury_rate",
    "global_policy_uncertainty", "deep_sea_freight_ppi"
]

In [6]:
monthly_external = model_df[
    ["date"] + external_cols
].drop_duplicates()

monthly_external = monthly_external.sort_values("date")

In [7]:
for col in external_cols:
    monthly_external[col + "_lag1"] = (
        monthly_external[col].shift(1)
    )

In [8]:
monthly_external.head()

,date,crude_oil_avg_usd,energy_index,non_energy_index,food_index,fertilizers_index,base_metals_index,precious_metals_index,fed_funds_rate,usd_broad_index,...,fertilizers_index_lag1,base_metals_index_lag1,precious_metals_index_lag1,fed_funds_rate_lag1,usd_broad_index_lag1,us_industrial_production_lag1,us_cpi_lag1,us_10y_treasury_rate_lag1,global_policy_uncertainty_lag1,deep_sea_freight_ppi_lag1
0,2014-01-01,102.0967,131.2182,97.0601,106.5227,105.6653,88.1316,100.5996,0.07,94.464748,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-02-01,104.8267,133.9666,97.9067,108.2873,105.6094,86.8503,105.0007,0.07,94.645582,...,105.6653,88.1316,100.5996,0.07,94.464748,100.1599,235.288,2.858095,115.257559,263.0
4,2014-03-01,104.0400,130.6676,99.9456,113.8253,102.5986,84.4461,107.2837,0.08,94.440821,...,105.6094,86.8503,105.0007,0.07,94.645582,100.9168,235.547,2.709474,101.147273,262.2
6,2014-04-01,104.8667,130.6783,99.9672,112.3142,94.9084,87.0879,103.9296,0.09,93.995550,...,102.5986,84.4461,107.2837,0.08,94.440821,101.8942,236.028,2.723333,114.652379,262.1
8,2014-05-01,105.7133,132.3273,99.5266,112.0518,96.0311,88.5446,102.9955,0.09,93.670029,...,94.9084,87.0879,103.9296,0.09,93.995550,101.9928,236.468,2.705238,110.450150,263.3


In [9]:
lag_columns = [
    "date"
] + [col + "_lag1" for col in external_cols]

In [10]:
monthly_external_lagged = monthly_external[
    lag_columns
].copy()

In [11]:
monthly_external_lagged.head()

,date,crude_oil_avg_usd_lag1,energy_index_lag1,non_energy_index_lag1,food_index_lag1,fertilizers_index_lag1,base_metals_index_lag1,precious_metals_index_lag1,fed_funds_rate_lag1,usd_broad_index_lag1,us_industrial_production_lag1,us_cpi_lag1,us_10y_treasury_rate_lag1,global_policy_uncertainty_lag1,deep_sea_freight_ppi_lag1
0,2014-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-02-01,102.0967,131.2182,97.0601,106.5227,105.6653,88.1316,100.5996,0.07,94.464748,100.1599,235.288,2.858095,115.257559,263.0
4,2014-03-01,104.8267,133.9666,97.9067,108.2873,105.6094,86.8503,105.0007,0.07,94.645582,100.9168,235.547,2.709474,101.147273,262.2
6,2014-04-01,104.0400,130.6676,99.9456,113.8253,102.5986,84.4461,107.2837,0.08,94.440821,101.8942,236.028,2.723333,114.652379,262.1
8,2014-05-01,104.8667,130.6783,99.9672,112.3142,94.9084,87.0879,103.9296,0.09,93.995550,101.9928,236.468,2.705238,110.450150,263.3


In [12]:
model_df = pd.merge(
    model_df,
    monthly_external_lagged,
    on="date",
    how="left"
)

In [13]:
print(model_df.shape)

(4668, 37)


In [14]:
model_df = model_df.dropna(
    subset=[col + "_lag1" for col in external_cols]
)

In [15]:
print(model_df.shape)

(4636, 37)


In [16]:
model_df = model_df.drop(
    columns=external_cols
)

In [17]:
print(model_df.shape)

(4636, 23)


In [18]:
model_df["log_primaryValue"] = np.log1p(
    model_df["primaryValue"]
)

In [19]:
model_df[
    ["primaryValue", "log_primaryValue"]
].head()

,primaryValue,log_primaryValue
2,5.539307e+09,22.435135
3,4.647254e+09,22.259542
4,5.524368e+09,22.432435
5,4.913932e+09,22.315340
6,5.414486e+09,22.412344


In [20]:
model_df = model_df.drop(
    columns=["trade_value_lag12", "lag12_date"],
    errors="ignore"
)

In [21]:
lag_lookup = model_df[
    ["country_name", "flowCode", "date", "primaryValue"]
].copy()

In [22]:
lag_lookup["date"] = (
    lag_lookup["date"] + pd.DateOffset(months=12)
)

lag_lookup = lag_lookup.rename(
    columns={"primaryValue": "trade_value_lag12"}
)

In [23]:
model_df = model_df.merge(
    lag_lookup,
    on=["country_name", "flowCode", "date"],
    how="left"
)

In [24]:
seasonal_model_df = model_df.dropna(
    subset=["trade_value_lag12"]
).copy()

In [25]:
print(seasonal_model_df.shape)

(4196, 25)


In [26]:
seasonal_model_df_encoded = pd.get_dummies(
    seasonal_model_df,
    columns=["country_name", "flowCode"],
    drop_first=True
)

In [27]:
print(seasonal_model_df_encoded.shape)

(4196, 41)


In [28]:
seasonal_model_df_encoded.to_csv(
    "seasonal_model_df_encoded.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [29]:
import os

print(os.path.exists("seasonal_model_df_encoded.csv"))

True
